# **Benchmarking**
In this notebook, we will be comparing our algorithm to other state of art algorithms

## **1. Imports and Benchmarking**

### **1.1 Imports**

In [17]:
import torch
import numpy as np
import torchvision
import matplotlib.pyplot as plt
import kagglehub
from transformers import AutoImageProcessor, AutoModel
import os
from torch.utils.data import DataLoader, Dataset

### **1.2 Model Prep**

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
model_name = "facebook/dinov3-vitl16-pretrain-lvd1689m"
#model_name = "facebook/dinov3-vit7b16-pretrain-lvd1689m" # Flagship

print(f"Loading flagship model: {model_name}")
print(f"Targeting Architecture: 40 Layers, 4096 Embedding Dim")

if 'dtype' not in locals():
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

try:
    #Load Model with Memory Optimization and Eager Attention
    # We strictly need eager attention to capture the attention weights
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
        attn_implementation="eager"
    ).to(device).eval()

    #Load Processor
    processor = AutoImageProcessor.from_pretrained(model_name)

    num_registers = getattr(model.config, "num_register_tokens", 4)
    print(f"Model loaded. Number of register tokens identified: {num_registers}")

    transform = lambda img: processor(images=img, return_tensors="pt")['pixel_values'].squeeze()

    #Verify Layer Count
    if hasattr(model, "vision_model"):
        num_layers = len(model.vision_model.encoder.layers)
    else:
        #Fallback for flat architectures
        num_layers = len(model.layer)

    print(f"Success! Model loaded with {num_layers} layers.")

except Exception as e:
    print(f"Error loading model: {e}")
    print("Tip: If you run out of memory (OOM), try 'dinov3-vitl16' (Large, 24 layers) instead.")

Loading flagship model: facebook/dinov3-vitl16-pretrain-lvd1689m
Targeting Architecture: 40 Layers, 4096 Embedding Dim


config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

Model loaded. Number of register tokens identified: 4
Success! Model loaded with 24 layers.


# **2. Dataset Download**

In [6]:
os.makedirs('./data', exist_ok=True)

### **2.1 ID**

In [ ]:
cifar10_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar10_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
print(f"✅ CIFAR-10 Loaded: {len(cifar10_train)} Train | {len(cifar10_test)} Test")

In [8]:
cifar100_train = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
cifar100_test = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
print(f"✅ CIFAR-100 Loaded: {len(cifar100_train)} Train | {len(cifar100_test)} Test")


  0%|          | 0.00/169M [00:00<?, ?B/s]
  0%|          | 32.8k/169M [00:00<18:07, 155kB/s]
  0%|          | 65.5k/169M [00:00<18:02, 156kB/s]
  0%|          | 98.3k/169M [00:00<17:58, 157kB/s]
  0%|          | 229k/169M [00:00<08:14, 341kB/s] 
  0%|          | 459k/169M [00:01<04:35, 611kB/s]
  1%|          | 918k/169M [00:01<02:26, 1.15MB/s]
  1%|          | 1.44M/169M [00:01<01:47, 1.56MB/s]
  2%|▏         | 3.15M/169M [00:01<00:45, 3.67MB/s]
  2%|▏         | 4.16M/169M [00:01<00:40, 4.04MB/s]
  3%|▎         | 5.21M/169M [00:02<00:37, 4.33MB/s]
  4%|▎         | 6.29M/169M [00:02<00:35, 4.57MB/s]
  4%|▍         | 7.05M/169M [00:02<00:38, 4.17MB/s]
  5%|▍         | 8.39M/169M [00:02<00:32, 4.94MB/s]
  5%|▌         | 9.18M/169M [00:02<00:34, 4.58MB/s]
  6%|▌         | 9.99M/169M [00:03<00:36, 4.36MB/s]
  6%|▋         | 10.8M/169M [00:03<00:37, 4.22MB/s]
  7%|▋         | 11.7M/169M [00:03<00:37, 4.16MB/s]
  7%|▋         | 12.5M/169M [00:03<00:37, 4.12MB/s]
  8%|▊         | 13.4M/169M

✅ CIFAR-100 Loaded: 50000 Train | 10000 Test


### **2.2 OOD**

In [9]:
svhn_test = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
print(f"✅ SVHN (OOD) Loaded: {len(svhn_test)} Test images")


  0%|          | 0.00/64.3M [00:00<?, ?B/s]
  0%|          | 32.8k/64.3M [00:00<15:25, 69.4kB/s]
  0%|          | 65.5k/64.3M [00:00<10:54, 98.2kB/s]
  0%|          | 98.3k/64.3M [00:00<09:27, 113kB/s] 
  0%|          | 131k/64.3M [00:01<08:46, 122kB/s] 
  0%|          | 164k/64.3M [00:01<08:23, 127kB/s]
  0%|          | 197k/64.3M [00:01<08:10, 131kB/s]
  0%|          | 262k/64.3M [00:01<06:02, 177kB/s]
  1%|          | 328k/64.3M [00:02<05:08, 207kB/s]
  1%|          | 393k/64.3M [00:02<04:39, 229kB/s]
  1%|          | 492k/64.3M [00:02<03:43, 285kB/s]
  1%|          | 590k/64.3M [00:02<03:16, 324kB/s]
  1%|          | 688k/64.3M [00:03<03:00, 352kB/s]
  1%|▏         | 819k/64.3M [00:03<02:33, 412kB/s]
  1%|▏         | 950k/64.3M [00:03<02:19, 454kB/s]
  2%|▏         | 1.11M/64.3M [00:03<02:00, 525kB/s]
  2%|▏         | 1.31M/64.3M [00:04<01:42, 616kB/s]
  2%|▏         | 1.54M/64.3M [00:04<01:26, 722kB/s]
  3%|▎         | 1.80M/64.3M [00:04<01:14, 837kB/s]
  3%|▎         | 2.10M/64.

✅ SVHN (OOD) Loaded: 26032 Test images


In [10]:
dtd_test = torchvision.datasets.DTD(root='./data', split='test', download=True, transform=transform)
print(f"✅ DTD Textures (OOD) Loaded: {len(dtd_test)} Test images")


  0%|          | 0.00/625M [00:00<?, ?B/s]
  0%|          | 32.8k/625M [00:00<53:38, 194kB/s]
  0%|          | 98.3k/625M [00:00<34:13, 304kB/s]
  0%|          | 164k/625M [00:00<30:27, 342kB/s] 
  0%|          | 360k/625M [00:00<15:44, 662kB/s]
  0%|          | 754k/625M [00:00<08:18, 1.25MB/s]
  0%|          | 1.54M/625M [00:01<04:20, 2.39MB/s]
  0%|          | 3.05M/625M [00:01<02:18, 4.49MB/s]
  1%|          | 6.13M/625M [00:01<01:10, 8.80MB/s]
  2%|▏         | 9.90M/625M [00:01<00:47, 12.9MB/s]
  2%|▏         | 13.9M/625M [00:01<00:38, 16.0MB/s]
  3%|▎         | 18.0M/625M [00:01<00:33, 18.1MB/s]
  4%|▎         | 22.0M/625M [00:02<00:30, 19.5MB/s]
  4%|▍         | 26.0M/625M [00:02<00:29, 20.5MB/s]
  5%|▍         | 30.0M/625M [00:02<00:28, 21.2MB/s]
  5%|▌         | 34.0M/625M [00:02<00:27, 21.6MB/s]
  6%|▌         | 38.1M/625M [00:02<00:26, 22.0MB/s]
  7%|▋         | 42.1M/625M [00:02<00:26, 22.3MB/s]
  7%|▋         | 46.1M/625M [00:03<00:25, 22.3MB/s]
  8%|▊         | 50.1M/625

✅ DTD Textures (OOD) Loaded: 1880 Test images


In [11]:
places365_test = torchvision.datasets.Places365(
        root='./data',
        split='val',
        small=True,
        download=True,
        transform=transform
    )
print(f"✅ Places365 (OOD) Loaded: {len(places365_test)} images")


  0%|          | 0.00/67.5M [00:00<?, ?B/s]
  0%|          | 32.8k/67.5M [00:00<07:45, 145kB/s]
  0%|          | 65.5k/67.5M [00:00<07:47, 144kB/s]
  0%|          | 98.3k/67.5M [00:00<07:48, 144kB/s]
  0%|          | 229k/67.5M [00:00<03:34, 314kB/s] 
  1%|          | 426k/67.5M [00:01<02:11, 511kB/s]
  1%|          | 786k/67.5M [00:01<01:16, 874kB/s]
  2%|▏         | 1.51M/67.5M [00:01<00:40, 1.62MB/s]
  4%|▍         | 2.92M/67.5M [00:01<00:21, 3.07MB/s]
  8%|▊         | 5.64M/67.5M [00:02<00:10, 5.83MB/s]
 13%|█▎        | 8.45M/67.5M [00:02<00:07, 7.84MB/s]
 17%|█▋        | 11.1M/67.5M [00:02<00:06, 9.05MB/s]
 21%|██        | 13.9M/67.5M [00:02<00:05, 9.96MB/s]
 25%|██▍       | 16.6M/67.5M [00:02<00:04, 10.5MB/s]
 29%|██▊       | 19.3M/67.5M [00:03<00:04, 10.9MB/s]
 33%|███▎      | 22.0M/67.5M [00:03<00:04, 11.2MB/s]
 37%|███▋      | 24.8M/67.5M [00:03<00:03, 11.5MB/s]
 41%|████      | 27.6M/67.5M [00:03<00:03, 11.7MB/s]
 45%|████▍     | 30.3M/67.5M [00:04<00:03, 11.8MB/s]
 49%|████

✅ Places365 (OOD) Loaded: 36500 images


### **2.3 Dataloaders**

In [15]:
BATCH_SIZE = 16

In [18]:
id_train_loader = DataLoader(cifar100_train, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
id_test_loader = DataLoader(cifar100_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [19]:
svhn_loader = DataLoader(svhn_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
dtd_loader = DataLoader(dtd_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
places_loader = DataLoader(places365_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

### **2.4 Synthetic Noise**

In [20]:
from PIL import Image

In [21]:
class SyntheticNoiseDataset(Dataset):
    def __init__(self, size=10000, img_size=(32, 32, 3), noise_type='gaussian', transform=None):
        """
        Generates 10,000 synthetic noise images to match standard benchmark sizes.
        """
        self.size = size
        self.img_size = img_size
        self.noise_type = noise_type
        self.transform = transform

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if self.noise_type == 'gaussian':
            # Mean 128, Std 64, clipped to 0-255
            arr = np.random.normal(128, 64, self.img_size)
            arr = np.clip(arr, 0, 255).astype(np.uint8)
        elif self.noise_type == 'uniform':
            # Uniformly distributed between 0-255
            arr = np.random.uniform(0, 255, self.img_size).astype(np.uint8)

        img = Image.fromarray(arr)

        if self.transform:
            img = self.transform(img)

        return img, -1 # -1 label signifies OOD

### **2.5 Synthetic Noise**

In [22]:
gaussian_dataset = SyntheticNoiseDataset(noise_type='gaussian', transform=transform)
uniform_dataset = SyntheticNoiseDataset(noise_type='uniform', transform=transform)

gaussian_loader = DataLoader(gaussian_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
uniform_loader = DataLoader(uniform_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## **3. Feature Extraction**

In [23]:
os.makedirs('./features', exist_ok=True)

### **3.1 Function definition**

In [24]:
@torch.no_grad()
def extract_and_save_features(model, dataloader, device, save_path):
    if os.path.exists(save_path):
        print(f"⏭️ Skipping {save_path} (Already exists)")
        return

    model.eval()
    all_features = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc=f"Extracting -> {os.path.basename(save_path)}"):
        images = images.to(device)

        # Forward pass
        outputs = model(pixel_values=images)

        # Extract the [CLS] token (global image representation)
        cls_tokens = outputs.last_hidden_state[:, 0, :].cpu()

        all_features.append(cls_tokens)
        all_labels.append(labels)

    all_features = torch.cat(all_features, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    torch.save({'features': all_features, 'labels': all_labels}, save_path)
    print(f"✅ Saved {len(all_features)} features to {save_path}")

### **3.2 ID Extraction**

In [26]:
from tqdm.notebook import tqdm

In [27]:
extract_and_save_features(model, id_train_loader, device, './features/id_cifar100_train.pt')
extract_and_save_features(model, id_test_loader, device, './features/id_cifar100_test.pt')

Extracting -> id_cifar100_train.pt:   0%|          | 0/3125 [00:00<?, ?it/s]

✅ Saved 50000 features to ./features/id_cifar100_train.pt


Extracting -> id_cifar100_test.pt:   0%|          | 0/625 [00:00<?, ?it/s]

✅ Saved 10000 features to ./features/id_cifar100_test.pt


### **3.3 OOD Extraction**

In [28]:
extract_and_save_features(model, svhn_loader, device, './features/ood_svhn.pt')
extract_and_save_features(model, dtd_loader, device, './features/ood_dtd.pt')
extract_and_save_features(model, places_loader, device, './features/ood_places.pt')
extract_and_save_features(model, gaussian_loader, device, './features/ood_gaussian.pt')
extract_and_save_features(model, uniform_loader, device, './features/ood_uniform.pt')

Extracting -> ood_svhn.pt:   0%|          | 0/1627 [00:00<?, ?it/s]

✅ Saved 26032 features to ./features/ood_svhn.pt


Extracting -> ood_dtd.pt:   0%|          | 0/118 [00:00<?, ?it/s]

✅ Saved 1880 features to ./features/ood_dtd.pt


Extracting -> ood_places.pt:   0%|          | 0/2282 [00:00<?, ?it/s]

✅ Saved 36500 features to ./features/ood_places.pt


Extracting -> ood_gaussian.pt:   0%|          | 0/625 [00:00<?, ?it/s]

✅ Saved 10000 features to ./features/ood_gaussian.pt


Extracting -> ood_uniform.pt:   0%|          | 0/625 [00:00<?, ?it/s]

✅ Saved 10000 features to ./features/ood_uniform.pt


## **4. Loading the data**

In [ ]:
!pip install faiss-cpu

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_curve, roc_auc_score, average_precision_score
import faiss

In [ ]:
NUM_CLUSTERS = 1024          # Size of our Semantic Visual Vocabulary (K-Means)
K_NN = 7                     # Number of neighbors for classification
CHAOS_THRESHOLD = 4          # Minimum unique classes in NNs to flag as OOD
DISTANCE_THRESHOLD = 0.5     # Chi-Square distance threshold for OOD

In [ ]:
NUM_REGISTERS = 4
START_PATCH_IDX = 1 + NUM_REGISTERS

In [ ]:
FEATURES_DIR = './patch_features'
HISTOGRAMS_DIR = './histograms'
os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(HISTOGRAMS_DIR, exist_ok=True)

### **4.1 K-Means Patches Vectors**

In [38]:
os.makedirs('./histograms', exist_ok=True)

In [37]:
@torch.no_grad()
def collect_kmeans_samples(model, dataloader, max_samples, device):
    model.eval()
    collected_patches = []
    total_collected = 0

    for images, _ in tqdm(dataloader, desc="Gathering Patches for K-Means"):
        images = images.to(device)
        outputs = model(pixel_values=images)

        # Extract spatial patches: shape (batch_size, num_patches, hidden_dim)
        patches = outputs.last_hidden_state[:, START_PATCH_IDX:, :].cpu()

        # Flatten the batch and patch dimensions: shape (batch_size * num_patches, hidden_dim)
        patches = patches.reshape(-1, patches.shape[-1])

        collected_patches.append(patches)
        total_collected += patches.shape[0]

        if total_collected >= max_samples:
            break

    # Concatenate and randomly shuffle the exact amount we need
    collected_patches = torch.cat(collected_patches, dim=0)
    indices = torch.randperm(collected_patches.shape[0])
    return collected_patches[indices][:max_samples].numpy()

In [39]:
print(f"📥 Collecting ~{MAX_KMEANS_SAMPLES} patch features from ID Training Set...")
# Assuming id_train_loader is your CIFAR-100 or Aircraft training loader
train_sample = collect_kmeans_samples(model, id_train_loader, MAX_KMEANS_SAMPLES, device)

📥 Collecting ~300000 patch features from ID Training Set...


Gathering Patches for K-Means:   0%|          | 0/3125 [00:00<?, ?it/s]

In [41]:
d = train_sample.shape[1]
print(f"\n🧠 Training FAISS K-Means with {NUM_CLUSTERS} clusters on {train_sample.shape[0]} patches...")

# Enable GPU for FAISS if available (makes it ~10x faster)
use_gpu = torch.cuda.is_available()
kmeans = faiss.Kmeans(d=d, k=NUM_CLUSTERS, niter=20, verbose=True, gpu=use_gpu)
kmeans.train(train_sample)

centroids = kmeans.centroids

# Create our fast quantizer (this maps new patches to the nearest centroid)
quantizer = faiss.IndexFlatL2(d)
quantizer.add(centroids)
print("✅ Visual Vocabulary Built successfully!")


🧠 Training FAISS K-Means with 1024 clusters on 300000 patches...
✅ Visual Vocabulary Built successfully!


### **4.2 Histogams**

In [40]:
@torch.no_grad()
def extract_and_save_histograms(model, dataloader, quantizer, device, save_path):
    if os.path.exists(save_path):
        print(f"⏭️ Skipping {save_path} (Already exists)")
        return

    model.eval()
    all_histograms = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc=f"Building Histograms -> {os.path.basename(save_path)}"):
        images = images.to(device)
        outputs = model(pixel_values=images)

        # Extract patches for the batch
        batch_patches = outputs.last_hidden_state[:, START_PATCH_IDX:, :].cpu().numpy()

        # Process each image individually to build its histogram
        for i in range(batch_patches.shape[0]):
            img_patches = batch_patches[i] # shape (num_patches, hidden_dim)

            # Find nearest cluster for every patch in the image
            _, I = quantizer.search(img_patches, 1)

            # Build frequency histogram (bins=1024)
            hist, _ = np.histogram(I, bins=np.arange(NUM_CLUSTERS + 1))

            # L1 Normalize (Convert counts to percentages)
            hist = hist.astype(float) / (hist.sum() + 1e-10)

            all_histograms.append(hist)
            all_labels.append(labels[i].item())

    # Save the lightweight histograms to disk
    torch.save({
        'histograms': np.array(all_histograms),
        'labels': np.array(all_labels)
    }, save_path)
    print(f"✅ Saved {len(all_histograms)} histograms to {save_path}")

In [42]:
extract_and_save_histograms(model, id_train_loader, quantizer, device, './histograms/id_train_hists.pt')
extract_and_save_histograms(model, id_test_loader, quantizer, device, './histograms/id_test_hists.pt')

Building Histograms -> id_train_hists.pt:   0%|          | 0/3125 [00:00<?, ?it/s]

✅ Saved 50000 histograms to ./histograms/id_train_hists.pt


Building Histograms -> id_test_hists.pt:   0%|          | 0/625 [00:00<?, ?it/s]

✅ Saved 10000 histograms to ./histograms/id_test_hists.pt


In [43]:
# Extract OOD Histograms (Make sure you have these dataloaders defined from earlier!)
extract_and_save_histograms(model, svhn_loader, quantizer, device, './histograms/ood_svhn_hists.pt')
extract_and_save_histograms(model, dtd_loader, quantizer, device, './histograms/ood_dtd_hists.pt')
extract_and_save_histograms(model, places_loader, quantizer, device, './histograms/ood_places_hists.pt')

Building Histograms -> ood_svhn_hists.pt:   0%|          | 0/1627 [00:00<?, ?it/s]

✅ Saved 26032 histograms to ./histograms/ood_svhn_hists.pt


Building Histograms -> ood_dtd_hists.pt:   0%|          | 0/118 [00:00<?, ?it/s]

✅ Saved 1880 histograms to ./histograms/ood_dtd_hists.pt


Building Histograms -> ood_places_hists.pt:   0%|          | 0/2282 [00:00<?, ?it/s]

✅ Saved 36500 histograms to ./histograms/ood_places_hists.pt


## **5. Histograms K-NN**

In [29]:
NUM_CLUSTERS = 1024          # Size of our Semantic Visual Vocabulary (K-Means)
K_NN = 7                     # Number of neighbors for classification
CHAOS_THRESHOLD = 4          # Minimum unique classes in NNs to flag as OOD
DISTANCE_THRESHOLD = 0.5     # Chi-Square distance threshold for OOD

In [30]:
NUM_REGISTERS = 4
START_PATCH_IDX = 1 + NUM_REGISTERS

In [33]:
MAX_KMEANS_SAMPLES = 300000  # Max patches to randomly sample for training K-Means (prevents RAM crashes)

In [34]:
FEATURES_DIR = './patch_features'
HISTOGRAMS_DIR = './histograms'
os.makedirs(FEATURES_DIR, exist_ok=True)
os.makedirs(HISTOGRAMS_DIR, exist_ok=True)

### **5.1 Clustering**

In [ ]:
from collections import Counter

In [45]:
print("📥 Loading Histograms...")

# Add weights_only=False to bypass the new PyTorch 2.6+ security default
train_data = torch.load('./histograms/id_train_hists.pt', weights_only=False)
test_data  = torch.load('./histograms/id_test_hists.pt', weights_only=False)
ood_data   = torch.load('./histograms/ood_svhn_hists.pt', weights_only=False)

X_train = train_data['histograms']
y_train = train_data['labels']

X_test = test_data['histograms']
y_test = test_data['labels']

X_ood = ood_data['histograms']

📥 Loading Histograms...


In [47]:
K_NN = 7

In [ ]:
DISTANCE_THRESHOLD = 0.5
CHAOS_THRESHOLD = 4

In [46]:
def chi_square_distance(A, B):
    # Add a tiny epsilon (1e-10) to prevent division by zero
    return 0.5 * np.sum((A - B)**2 / (A + B + 1e-10))

In [ ]:
print(f"🧠 Fitting {K_NN}-NN Classifier with Chi-Square metric...")
nn = NearestNeighbors(n_neighbors=K_NN, metric=chi_square_distance, algorithm='brute')
nn.fit(X_train)

print("🔍 Querying ID Test Set...")
id_dist_matrix, id_idx_matrix = nn.kneighbors(X_test)

print("🔍 Querying OOD Set...")
ood_dist_matrix, ood_idx_matrix = nn.kneighbors(X_ood)

🧠 Fitting 7-NN Classifier with Chi-Square metric...
🔍 Querying ID Test Set...


In [ ]:
# Extract Distance to the 1st Nearest Neighbor
id_dist = id_dist_matrix[:, 0]
ood_dist = ood_dist_matrix[:, 0]

# Extract Neighborhood Chaos (Unique classes among the K nearest neighbors)
id_nn_labels = y_train[id_idx_matrix]
ood_nn_labels = y_train[ood_idx_matrix]

id_chaos = np.array([len(np.unique(row)) for row in id_nn_labels])
ood_chaos = np.array([len(np.unique(row)) for row in ood_nn_labels])

### **5.2 AUROC, ROC, FTPR**

In [ ]:
y_true = np.concatenate([np.zeros(len(id_dist)), np.ones(len(ood_dist))])
y_scores = np.concatenate([id_dist, ood_dist])

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
idx_95 = np.where(tpr >= 0.95)[0][0]
fpr95 = fpr[idx_95]
detection_error = 0.5 * (1 - 0.95) + 0.5 * fpr95

auroc = roc_auc_score(y_true, y_scores)
aupr_out = average_precision_score(y_true, y_scores)
aupr_in = average_precision_score(1 - y_true, -y_scores)

In [ ]:
id_flagged_ood = (id_dist > DISTANCE_THRESHOLD) | (id_chaos >= CHAOS_THRESHOLD)
ood_flagged_ood = (ood_dist > DISTANCE_THRESHOLD) | (ood_chaos >= CHAOS_THRESHOLD)

total_id = len(id_dist)
total_ood = len(ood_dist)
false_alarms_id = np.sum(id_flagged_ood)
correctly_flagged_ood = np.sum(ood_flagged_ood)
missed_ood = total_ood - correctly_flagged_ood

id_predictions = np.array([Counter(row).most_common(1)[0][0] for row in id_nn_labels])
correct_id_classifications = np.sum((id_predictions == y_test) & (~id_flagged_ood))

missed_ood_assigned_classes = [Counter(row).most_common(1)[0][0] for i, row in enumerate(ood_nn_labels) if not ood_flagged_ood[i]]

### **5.3 Plotting**

In [ ]:
import matplotlib.patches as patches

In [ ]:
def get_class_name(cls_id):
    return f"Class {int(cls_id)}"

In [ ]:
print("========================================================")
print("🛡️ OOD DETECTION PERFORMANCE:")
print(f"   -> Alien Catch Rate (Recall):  {(correctly_flagged_ood/total_ood)*100:.2f}% ({correctly_flagged_ood}/{total_ood} caught)")
print(f"   -> Missed Aliens (Slipped in): {missed_ood} images")
print(f"   -> False Alarms (ID flagged):  {false_alarms_id} images ({(false_alarms_id/total_id)*100:.2f}%)")
print("========================================================")
print("🎯 IN-DISTRIBUTION (ID) PERFORMANCE:")
id_passed = total_id - false_alarms_id
if id_passed > 0:
    print(f"   -> ID Accuracy (of those that passed): {(correct_id_classifications/id_passed)*100:.2f}% ({correct_id_classifications}/{id_passed})")
print("========================================================")

if missed_ood > 0:
    print("🕵️ WHAT DID THE MISSED ALIENS SNEAK IN AS?")
    top_missed = Counter(missed_ood_assigned_classes).most_common(5)
    for cls_id, count in top_missed:
        print(f"   -> Disguised as '{get_class_name(cls_id)}': {count} times")
print("========================================================")

### **5.4 Plotting**

In [ ]:
plot_data = {
    "id_chaos": id_chaos,
    "id_dist": id_dist,
    "ood_chaos": ood_chaos,
    "ood_dist": ood_dist
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

max_dist = max(np.max(id_dist), np.max(ood_dist)) * 1.1
rect = patches.Rectangle((CHAOS_THRESHOLD, DISTANCE_THRESHOLD),
                         K_NN - CHAOS_THRESHOLD + 0.5, max_dist - DISTANCE_THRESHOLD,
                         linewidth=1, edgecolor='red', facecolor='red', alpha=0.1, label='OOD Flag Zone')
ax1.add_patch(rect)

jitter_id = np.random.uniform(-0.15, 0.15, len(plot_data["id_chaos"]))
jitter_ood = np.random.uniform(-0.15, 0.15, len(plot_data["ood_chaos"]))

ax1.scatter(np.array(plot_data["id_chaos"]) + jitter_id, plot_data["id_dist"],
            color='blue', alpha=0.5, s=20, label='True ID (CIFAR-100)')
ax1.scatter(np.array(plot_data["ood_chaos"]) + jitter_ood, plot_data["ood_dist"],
            color='red', alpha=0.7, s=30, edgecolors='black', label='True Aliens (SVHN)')

ax1.axhline(DISTANCE_THRESHOLD, color='black', linestyle='--')
ax1.axvline(CHAOS_THRESHOLD, color='black', linestyle='--')

ax1.set_title("OOD Decision Boundaries\n(Distance vs. Neighborhood Chaos)", fontsize=15, fontweight='bold')
ax1.set_xlabel("Neighborhood Chaos (Unique Classes out of K)", fontsize=12)
ax1.set_ylabel("Chi-Square Distance (1-NN)", fontsize=12)
ax1.set_ylim(bottom=0, top=max_dist)
ax1.legend()
ax1.grid(alpha=0.3)

if missed_ood > 0:
    top_missed = Counter(missed_ood_assigned_classes).most_common(10)
    labels = [get_class_name(cls_id) for cls_id, _ in top_missed]
    counts = [count for _, count in top_missed]

    ax2.barh(labels[::-1], counts[::-1], color='orange', edgecolor='black')
    ax2.set_title("How the Missed Aliens Disguised Themselves\n(Assigned ID Classes)", fontsize=15, fontweight='bold')
    ax2.set_xlabel("Number of OOD Images assigned this class", fontsize=12)
    ax2.grid(axis='x', alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Perfect OOD Detection!\nNo Aliens Slipped Through.', ha='center', va='center', fontsize=18, color='green')
    ax2.axis('off')

plt.tight_layout()
plt.show()